# Building Firmware for the Acadia Control System

A major benefit of the Acadia architecture is its ability to be easily reconfigured with firmware images defined in Python, rather than in pure VHDL. This guide will walk through the build process for a Python-defined image and explain the various utilities provided for doing so.

First, we'll import the necessary libraries. The firmware we want to build is written as a class that we can directly import:

In [1]:
from acadia.system import Firmware

No module named 'pyxrfdc'
No module named 'pyxrfclk'


The first step is to create a directory for storing the Vivado project that we'll generate. We then instantiate our firmware object with this path (if the directory doesn't exist, this instantiation will create it):

In [2]:
project_dir = "/home/billy/acadia-build"

The firmware may define a number of custom VHDL modules. We need to write these to a file that the Vivado project can then import:

In [3]:
Firmware.write_hdl(project_dir)

Then, we need to create a TCL script that will populate the HEDGEHOG logic with the relevant objects for this type of firmware:

In [4]:
Firmware.write_hedgehog_tcl(project_dir)

Then, we run the provided project creation script in a terminal to command Vivado to create the project in our example directory and configure it with the files we just wrote:

```
vivado -mode tcl -source /home/billy/acadia/logic/make_project.tcl -tclargs --project_dir /home/billy/acadia-build --origin_dir /home/billy/acadia/logic/src
```

Then, begin the implementation with the following (credit to http://xillybus.com/tutorials/vivado-timing-constraints-error for automatic detection of timing failure):

```
launch_runs impl_1 -to_step write_bitstream -jobs 16
wait_on_run impl_1
if {[get_property PROGRESS [get_runs impl_1]] != "100%"} {
   error "ERROR: impl_1 failed"
   return -code error
}

set timing_report [report_timing_summary -no_header -no_detailed_paths -return_string]

if {! [string match -nocase {*timing constraints are met*} $timing_report]} {
    error "ERROR: timing not met"
    return -code error
}
```

Once it completes, export the hardware description file with the bitstream using the following:

```
write_hw_platform -fixed -include_bit -force -file /home/billy/acadia-build/acadia_bd_wrapper.xsa
```